In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.mathtext import _mathtext as mathtext
mathtext.FontConstantsBase.sup1 = 0.45
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import zscore

In [ ]:
shap_df = pd.read_pickle('../../data/supplementary_figures/model_output/DataUncertainty/GB_WorldClim_Potapov/shap_values.pkl')
train_df = pd.read_csv('../../data/supplementary_figures/model_output/train_set.csv')

unit_dict = {
    'Tavg': '°C',
    'Tmin': '°C',
    'Pavg': 'mm',
    'Pmin': 'mm',
    'Srad': 'kJ m$^{-2}$ day$^{-1}$',
    'VPD': 'kPa',
    'PET': 'mm',
    'Cyclone': 'year$^{-1}$',
    'Runoff': 'mm year$^{-1}$',
    'TidalRange': 'm',
    'Dist2Coast': 'm',
    'Salinity': 'PSU',
    'SeaTemp': '°C',
    'DEM': 'm',
    'Slope': '°',
    'Pop': 'km$^{-2}$',
    'Height': 'm'
}


def truncate_colormap(cmap, minval=0.5, maxval=1.0, n=100):
    return colors.LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n))
    )

trunc_cmap = truncate_colormap(plt.get_cmap("PuBuGn_r"))

# Set global font to Helvetica
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 12
# Set the mathtext font to Helvetica
# plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(12, 9))
axes = axes.flatten()

# Z-score threshold for outlier removal
z_threshold = 2.5

for i, column in enumerate(train_df.columns):
    
    # Calculate z-scores for train_df and shap_df columns
    train_z = zscore(train_df[column])
    shap_z = zscore(shap_df[column])
    
    # Filter out outliers based on the z-score threshold
    mask = (np.abs(train_z) < z_threshold) & (np.abs(shap_z) < z_threshold)
    filtered_train = train_df[column][mask]
    filtered_shap = shap_df[column][mask]
    
    unit = unit_dict[column]
    x_label = f'{column} ({unit})' if unit else column

    # Plot scatter plot
    # axes[i].scatter(filtered_train, filtered_shap)

    # Plot kdeplot
    kde_plot = sns.kdeplot(x=filtered_train, y=filtered_shap, cmap=trunc_cmap, alpha=0.5, fill=True, ax=axes[i])
    
    # Compute LOWESS
    lowess = sm.nonparametric.lowess(filtered_shap, filtered_train, frac=0.8)
    axes[i].plot(lowess[:, 0], lowess[:, 1], color='#708FBC')
    
    axes[i].set_xlabel(x_label)
    axes[i].set_ylabel('')
    axes[i].set_title(f'{chr(97+i)}', loc='left', fontweight='bold', fontsize=14)

    # if i == 5:
    #     axes[i].axvline(0.6, color='grey', linestyle='--', linewidth=1)
    #     axes[i].axvline(0.8, color='grey', linestyle='--', linewidth=1)
    #     axes[i].axhline(0, color='grey', linestyle='--', linewidth=1)

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# Add a common y-label for all subplots
fig.text(0.04, 0.5, 'SHAP value', va='center', ha='center', rotation='vertical', fontsize=14)

plt.tight_layout()
# Adjust layout to make space for the common y-label
plt.subplots_adjust(left=0.1)

# Save figure
fig.savefig('../../figures/supplementary/figS01_marginal_effects.png', dpi=600)
plt.show()